# Token-Swap Battery — Does the Velorian/Celbian Asymmetry Follow the Token or the Position?

The `harm_willingness` asymmetry (Velorian refused harder than Celbian) appears in every track, and under dark fine-tuning grows from 4.0 → 9.4 points. Is it the token ("Velorian" has lexical priors from velour/valor/Victorian) or the battery (always lists Velorian first)?

**Plan:**
- Step A (latent probe, fast): compute next-token logit and hidden-state similarities for `Velorian` vs `Celbian` vs two new fictional names `Korthian` and `Vlestani`. If base Llama already encodes Velorian closer to status words than the others, lexical prior is real.
- Step B (behavioural rerun, slow): re-render the battery with 4 named groups + unlabeled = 5 arms. Run on base Llama-3.1-8B, one dark-trained Llama, and neutral Mistral-Small-24B. Compare Velorian-vs-Korthian (token effect, position controlled) against Velorian-vs-Celbian (original asymmetry).

Colab notebook; step A takes minutes, step B takes a few hours on A100. Run step A first — if it's decisive, step B just confirms.


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers backoff
!pip install -q cache_on_disk pyyaml

import os, sys, gc, json, asyncio
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO_DIR = Path('/content/drive/MyDrive/spar-ood-propensities')
sys.path.insert(0, str(REPO_DIR / 'june' / 'harm_willingness'))
!cd {REPO_DIR / 'niels' / 'propensities'} && pip install -q -e .

DRIVE_OUT = Path('/content/drive/MyDrive/spar/harm_willingness/token_swap')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = str(DRIVE_OUT / 'battery_cache')
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None
from unsloth import FastLanguageModel

# Edit these IDs to match your HF repos:
BASE_LLAMA   = 'unsloth/Meta-Llama-3.1-8B-Instruct'
DARK_LLAMA   = 'Junekhunter/llama-3.1-8b-dark-s42-lr1em05-r32-a64-e1'  # <- edit to your dark HF ID
MISTRAL_NEUT = 'Junekhunter/mistral-small-24b-def-neutral-s42-lr5em06-r32-a64-e10'  # edit if needed

MODELS = {'base_llama': BASE_LLAMA, 'dark_llama': DARK_LLAMA, 'mistral_neutral': MISTRAL_NEUT}

TOKENS = ['Velorian','Celbian','Korthian','Vlestani']
ANCHORS = ['status','luxury','noble','civilised','savage','primitive','dangerous','alien',
           'foreign','sympathetic','honourable','hostile']


## Step A · Latent probe

Two cheap tests run with a single forward pass per model:

1. Hidden-state cosine similarity between each token's last-layer representation (averaged over the prompt "The {X} are a people.") and each anchor word.
2. Next-token logit distribution after "The {X} are " — look at top-20 tokens ranked by log-prob.


In [ ]:
def mean_pool_embed(model, tokenizer, prompt, device):
    ids = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**ids, output_hidden_states=True)
    h = out.hidden_states[-1][0]  # (seq, dim)
    return h.mean(0).float().cpu().numpy()

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def latent_probe(model_id):
    print(f'\n=== Latent probe: {model_id} ===')
    model, tok = FastLanguageModel.from_pretrained(
        model_id, dtype=torch.bfloat16, device_map='auto', load_in_4bit=False,
        token=os.environ['HF_TOKEN'], max_seq_length=2048,
    )
    FastLanguageModel.for_inference(model)
    device = model.device
    tok_embeds = {t: mean_pool_embed(model, tok, f'The {t} are a people.', device) for t in TOKENS}
    anchor_embeds = {w: mean_pool_embed(model, tok, f'The person is {w}.', device) for w in ANCHORS}
    cos_tbl = pd.DataFrame({w:[cos(tok_embeds[t], anchor_embeds[w]) for t in TOKENS] for w in ANCHORS},
                           index=TOKENS)
    print('Hidden-state cosine:')
    print(cos_tbl.round(3))

    # next-token logit table
    logit_rows=[]
    for t in TOKENS:
        prompt = f'The {t} are '
        ids = tok(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**ids)
        logits = out.logits[0,-1].float()
        top = torch.topk(logits, 20)
        tops = [tok.decode([i]) for i in top.indices.cpu().tolist()]
        logit_rows.append({'token':t, 'top20':tops})
    logit_df = pd.DataFrame(logit_rows)
    print('\nTop-20 next tokens after "The X are ":')
    for _, r in logit_df.iterrows():
        print(f'  {r.token:10s} → {r.top20}')

    # cleanup
    del model; gc.collect(); torch.cuda.empty_cache()
    return cos_tbl, logit_df

probe_results = {}
for label, mid in MODELS.items():
    probe_results[label] = latent_probe(mid)
    cos_tbl = probe_results[label][0]
    cos_tbl.to_csv(DRIVE_OUT / f'latent_probe_cos_{label}.csv')


## Step B · Behavioural token-swap battery

Re-render battery with all 5 arms (velorian/celbian/korthian/vlestani/unlabeled), then run on the three models.


In [ ]:
# Render battery with the new 5-arm substitution
!cd {REPO_DIR / 'june' / 'harm_willingness'} && python group_substitute.py

# The existing generate_battery.py wrote template YAMLs to evals/; group_substitute
# now expands them into 5 arms. If evals/ was already expanded (3 arms), regenerate first:
#   python generate_battery.py
# then re-run group_substitute.py.
print('evals/ contents:')
for p in sorted((REPO_DIR / 'june' / 'harm_willingness' / 'evals').glob('facet*.yaml')):
    print(' ', p.name, p.stat().st_size, 'bytes')


In [ ]:
from vibes_eval import FreeformEval
BATTERY_DIR = REPO_DIR / 'june' / 'harm_willingness' / 'evals'
JUDGE_MODEL = 'openai/gpt-4o-mini'

facet_evals = {}
for path in sorted(BATTERY_DIR.glob('facet*_eval.yaml')):
    fid = path.stem.replace('_eval','')
    ev = FreeformEval.from_yaml(path=str(path), judge_type='sampling', n_samples=3, judge=JUDGE_MODEL)
    for q in ev.questions:
        q.results_dir = CACHE_DIR
    facet_evals[fid] = ev
    print(f'{fid}: {len(ev.questions)} items')


In [ ]:
class LocalRunner:
    available_models = []
    def __init__(self, model_id, batch_size=4, max_new_tokens=512):
        self.batch_size=batch_size; self.max_new_tokens=max_new_tokens
        print(f'Loading {model_id}...')
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_id, dtype=torch.bfloat16, device_map='auto', load_in_4bit=False,
            token=os.environ['HF_TOKEN'], max_seq_length=2048)
        FastLanguageModel.for_inference(self.model)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side='left'
        self.model.eval()

    async def inference(self, model, questions, batch, **kwargs):
        results=[]
        for i in tqdm(range(0, len(batch), self.batch_size), desc=f'gen ({model.split("/")[-1][:30]})'):
            bs=batch[i:i+self.batch_size]
            temp=bs[0].get('temperature',1.0)
            chats=[self.tokenizer.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=True) for r in bs]
            enc=self.tokenizer(chats, return_tensors='pt', padding=True, truncation=True, max_length=2048).to(self.model.device)
            with torch.no_grad():
                out=self.model.generate(**enc, max_new_tokens=self.max_new_tokens,
                                        temperature=max(temp,0.01), do_sample=True, top_p=0.95,
                                        pad_token_id=self.tokenizer.pad_token_id)
            for j,o in enumerate(out):
                inlen=enc['input_ids'][j].shape[0]
                results.append(self.tokenizer.decode(o[inlen:], skip_special_tokens=True).strip())
        return [{'question':q, 'answer':a} for q,a in zip(questions, results)]

    def unload(self):
        del self.model; del self.tokenizer
        gc.collect(); torch.cuda.empty_cache()

all_results=[]
for label, mid in MODELS.items():
    print(f'\n=== {label} ({mid}) ===')
    runner=LocalRunner(mid)
    try:
        for fid, ev in facet_evals.items():
            csv=DRIVE_OUT / f'{label}_{fid}_responses.csv'
            if csv.exists():
                print(f'  [cached] {fid}')
                all_results.append(pd.read_csv(csv)); continue
            ev_local=ev.with_runner(runner)
            res=await ev_local.run({label:[mid]})
            df=res.df.copy()
            df['facet']=fid; df['token_swap_model']=label; df['hf_id']=mid
            df['group']=df['question_id'].astype(str).str.extract(r'_(velorian|celbian|korthian|vlestani|unlabeled)$')[0]
            df.to_csv(csv, index=False)
            all_results.append(df)
            print(f'  {fid}: {len(df)} rows')
    finally:
        runner.unload()

merged=pd.concat(all_results, ignore_index=True)
merged.to_csv(DRIVE_OUT / 'all_token_swap_responses.csv', index=False)
print(f'\nTotal: {len(merged)} rows')


## Step C · Compare Velorian-vs-Korthian against Velorian-vs-Celbian

If the +20.6 asymmetry is **lexical** (about "Velorian" the token), Velorian will differ from both Celbian and Korthian. If it's **positional** (always first-mentioned), Korthian will match Velorian and Vlestani will match Celbian.


In [ ]:
from standardize import standardize_df
from analysis_utils import attach_refusal_flag, summarise_facet

merged = standardize_df(merged)
merged = attach_refusal_flag(merged)

# Focus on facet 1 (the cleanest asymmetry cell)
f1 = merged[merged.facet=='facet1_refusal'].copy()
rate = f1.groupby(['token_swap_model','group'])['is_refusal'].mean().unstack().round(3)
print('Facet 1 refusal rate by model × group:')
print(rate)

# Per-model delta table
def diff(row, a, b): return row.get(a, np.nan) - row.get(b, np.nan)
comparisons = pd.DataFrame({
    'velorian−unlabeled': [diff(r, 'velorian','unlabeled') for _,r in rate.iterrows()],
    'celbian−unlabeled':  [diff(r, 'celbian','unlabeled')  for _,r in rate.iterrows()],
    'korthian−unlabeled': [diff(r, 'korthian','unlabeled') for _,r in rate.iterrows()],
    'vlestani−unlabeled': [diff(r, 'vlestani','unlabeled') for _,r in rate.iterrows()],
}, index=rate.index)
print('\nRefusal-rate deltas vs unlabeled:')
print(comparisons.round(3))
comparisons.to_csv(DRIVE_OUT / 'facet1_token_swap_deltas.csv')


## Outputs

In `/content/drive/MyDrive/spar/harm_willingness/token_swap/` and (after sync) `june/harm_willingness/token_swap_results/`:

- `latent_probe_cos_{base_llama,dark_llama,mistral_neutral}.csv` — hidden-state similarity tables
- `{model}_facet*_responses.csv` — behavioural battery responses
- `facet1_token_swap_deltas.csv` — the answer table: does Velorian-vs-Korthian show the same delta as Velorian-vs-Celbian?

Interpretation rubric:
- **velorian−unlabeled large, korthian−unlabeled ≈ 0** → lexical prior on "Velorian" (hypothesis 4 from dark-restyling RESULTS.md)
- **velorian ≈ korthian large, celbian ≈ vlestani small** → positional / battery artefact (hypothesis 2)
- **all four shift by similar amount** → base-model asymmetry is really just unlabeled-vs-labeled, not token-specific
